In [3]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.autograd import grad

## Basic Overview
We are trying to solve the Allen-Cahn equation, but not numerically. We will try to see if a **Neural Network** can just learn the solution ϕ(x,t) directly.

### 1) Defining the Neural Network
We are defining the brain of our PINN (Physics Informed Neural Network) the neural network that will take a point (x,t) and output a predicted phase field value ϕ^​. 

Think of it as building a mathematical function approximator whose internal parameters (weights and biases) will be tuned later during training of the NN.

In [4]:
class PINN(nn.Module):
    def __init__(self, layers):     #__init__ is a constructor method that initializes NN architecture.  
        #self is specific instance of class
        #Layers store number of neurons in each layer, including input and output

        super(PINN, self).__init__() #super() refers to parent class (nn.Module) and allows us to call its methods.
        #Helps pytorch to track parameters.
        
        self.network = nn.Sequential() #attribute to hold layers in a sequence.

        for i in range(len(layers) - 1):      #1 less pair than no. of. layers
            self.network.add_module(f"linear_{i}", nn.Linear(layers[i], layers[i+1]))   #adds a layer to nn.Sequential container
            #computes y = xA^T + b, where A is weight matrix and b is bias vector
            #For nn.Linear(2,64) takes 2 inputs (x,t), produces 64 neurons 

            if i < len(layers) - 2:     #No activation after last layer
                self.network.add_module(f"tanh_{i}", nn.Tanh())     #Add Tanh activation function after each hidden layer, outputs (-1, 1), helps model learn complex patterns.
    

    def forward(self, x, t):         #forward (special pytorch method) defines how data flows thorugh network

        inputs = torch.cat([x, t], dim=1)  # shape (N, 2), inputs x and t are concatenated along (column) feature dimension (dim=1)
        return self.network(inputs)         # shape:(N, 1), passes concatenated inputs through the network to produce output .

device = torch.device("cpu")

layers = [2, 64, 64, 64, 64, 1]   #input size 2 (x,t), 4 hidden layers with 64 neurons each, output size 1 (ϕ)
model = PINN(layers).to(device)

print(model)
print(f"\nTotal trainable parameters: {sum(p.numel() for p in model.parameters())}")
#model.parameters() returns an iterator over all learnable tensors (weights and biases) in the network
#p.numel() no. of. elements in each parameter tensor, sum() gives total count of trainable parameters in the model.

PINN(
  (network): Sequential(
    (linear_0): Linear(in_features=2, out_features=64, bias=True)
    (tanh_0): Tanh()
    (linear_1): Linear(in_features=64, out_features=64, bias=True)
    (tanh_1): Tanh()
    (linear_2): Linear(in_features=64, out_features=64, bias=True)
    (tanh_2): Tanh()
    (linear_3): Linear(in_features=64, out_features=64, bias=True)
    (tanh_3): Tanh()
    (linear_4): Linear(in_features=64, out_features=1, bias=True)
  )
)

Total trainable parameters: 12737


## 2) Physics Informed Loss Function
For now the network exists but outputs garbage, basically random weights mean random predictions. 
We have to train it without any data. So what do we target to minimize?
We minimize how badly the network's output violates the Allen-Cahn equation. 
If ϕ​^ is the actual solution, then when you plug it into the PDE the result should be exactly zero everywhere. 
Any deviation from zero is our needed error.

Ww will have 3 error components:
1) PDE Residual loss (PDE) - Sample random (x,t) points inside the domain, then we compute first compute ∂ϕ^/∂t and ∂^2(ϕ^)/∂x^2. 
Putting these into AC equation we can check how far is it from zero.
2) Initial Condition Loss (ICL) -  At t=0, we know what ϕ should look like (for now its a sinusoidal ). Our network should match this perfectly (ideally), error is the mismatch between them.
3) In Boundary Condition Loss (BCL) - At the edges of the domain we will enforce Neumann boundary conditions. 
Basically the spatial derivative at the walls is zero.